In [119]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import math

In [120]:
class PreNormBlock(nn.Module):
    def __init__(self, d_model: int, sublayer: nn.Module, dropout: float = 0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.sublayer = sublayer
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, *args, **kwargs):
        normalized = self.norm(x)
        output = self.sublayer(normalized, *args, **kwargs)
        return x + self.dropout(output)


In [121]:
class PositionEmdedding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2) * 
            (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)

        pe[:, 1::2] = torch.cos(position * div_term[:pe[:, 1::2].shape[1]])

        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


In [122]:
class SelfAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1, is_causal=False):
        super().__init__()
        self.attention = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.is_causal=is_causal

    def forward(self, x, key_padding_mask=None):
        mask = None

        if self.is_causal:
            seq_len = x.size(1)
            mask = torch.triu(
                torch.ones(
                    seq_len, seq_len,
                    device=x.device,
                    dtype=torch.bool
                ),
                diagonal=1
            )
        output, _= self.attention(
            x, x, x,
            attn_mask=mask,
            key_padding_mask=key_padding_mask,
            need_weights=False
            )
        return output

In [123]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attention = PreNormBlock(
            d_model,
            SelfAttention(d_model, num_heads, dropout),
            dropout
        )

        self.ffn = PreNormBlock(
            d_model, 
            nn.Sequential(
                nn.Linear(d_model, d_ff),
                nn.ReLU(),
                nn.Linear(d_ff, d_model)
            ),
            dropout
        )

    def forward(self, x, src_padding_mask=None):
        x = self.attention(x, key_padding_mask=src_padding_mask)
        x = self.ffn(x)
        return x




In [129]:
class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, n=6, dropout=0.1):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.positional_embedding = PositionEmdedding(d_model)
        self.dropout = nn.Dropout(dropout)

        self.layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(n)
        ])

    def forward(self, x, src_padding_mask=None):
        x = self.embedding(x)
        x = self.positional_embedding(x)
        x = self.dropout(x)

        for layer in self.layers:
            x = layer(x, src_padding_mask=src_padding_mask)
        return x

In [125]:
class CrossAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        self.attention = nn.MultiheadAttention(
                    embed_dim=d_model,
                    num_heads=num_heads,
                    dropout=dropout,
                    batch_first=True
                )

    def forward(self, x, encoder_output, src_padding_mask=None):
        output, _ = self.attention(
            query=x,
            key = encoder_output,
            value = encoder_output,
            key_padding_mask=src_padding_mask,
            need_weights=False
        )

        return output

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attention = PreNormBlock(
            d_model, 
            SelfAttention(d_model, num_heads, dropout, is_causal=True),
            dropout
        )

        self.attention_sec = PreNormBlock(
                    d_model, 
                    CrossAttention(d_model, num_heads, dropout),
                    dropout
                )

        self.ffn = PreNormBlock(
            d_model,
            nn.Sequential(
                nn.Linear(d_model, d_ff),
                nn.ReLU(),
                nn.Linear(d_ff, d_model)
            ),
            dropout
        )

    def forward(self, x, encoder_output, tgt_padding_mask=None, src_padding_mask=None):
        x = self.attention(x, key_padding_mask=tgt_padding_mask)
        x = self.attention_sec(x, encoder_output, src_padding_mask=src_padding_mask)
        x = self.ffn(x)
        return x

In [ ]:
class TransformerDecoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model,
        num_heads,
        d_ff,
        n=6,
        dropout=0.1,
        max_len=5000
    ):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.positional_embedding = PositionEmdedding(d_model, max_len)
        self.dropout = nn.Dropout(dropout)

        self.layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(n)
        ])

        self.output_projection = nn.Linear(d_model, vocab_size)

    def forward(self, x, encoder_output, tgt_padding_mask=None, src_padding_mask=None):
        x = self.embedding(x)
        x = self.positional_embedding(x)
        x = self.dropout(x)

        for layer in self.layers:
            x = layer(x, encoder_output, tgt_padding_mask=tgt_padding_mask, src_padding_mask=src_padding_mask)

        return self.output_projection(x)

In [ ]:
class Transformer(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt, src_padding_mask=None, tgt_padding_mask=None):
        encoder_output = self.encoder(
            src, 
            src_padding_mask=src_padding_mask)
        
        output = self.decoder(
            tgt,
            encoder_output, 
            tgt_padding_mask=tgt_padding_mask, 
            src_padding_mask=src_padding_mask)
        return output


